In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import os

os.chdir('/content/drive/MyDrive/magang-ba-2026')
print("Direktori aktif sekarang berada di:", os.getcwd())

Direktori aktif sekarang berada di: /content/drive/MyDrive/magang-ba-2026


**Hapus Kode Percobaan & Pastikan Setiap Fungsi di src/ Punya Docstring Satu Kalimat**

In [7]:
import os
import ast
import glob

print("Direktori kerja:", os.getcwd())

# 1. Cek semua file Python & file sementara di proyek
all_py = sorted(glob.glob("**/*.py", recursive=True))
print("\n=== DAFTAR FILE PYTHON DI PROYEK ===")
for f in all_py:
    print(" -", f)

temp_files = [
    f for f in glob.glob("**/*", recursive=True)
    if os.path.isfile(f) and any(k in os.path.basename(f).lower() for k in ["tmp", "temp", "coba", "test_coba", "untitled", ".bak", "~"])
]
print("\n=== FILE SEMENTARA / PERCOBAAN TERDETEKSI ===")
print(temp_files if temp_files else "Tidak ada file bernama sementara/percobaan.")

# 2. Cek docstring setiap fungsi di folder src/
src_files = sorted(glob.glob("src/**/*.py", recursive=True))
print(f"\n=== AUDIT DOCSTRING FUNGSI DI src/ ({len(src_files)} file) ===")
missing_docs = []

for fp in src_files:
    with open(fp, "r", encoding="utf-8") as f:
        tree = ast.parse(f.read(), filename=fp)
    for node in ast.walk(tree):
        if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)):
            doc = ast.get_docstring(node)
            if not doc:
                missing_docs.append((fp, node.lineno, node.name))

if missing_docs:
    print(f"Ditemukan {len(missing_docs)} fungsi di src/ tanpa docstring:")
    for fp, line, fn in missing_docs:
        print(f"  - {fp}:{line} -> def {fn}()")
else:
    print("Semua fungsi di src/ sudah memiliki docstring!")

Direktori kerja: /content/drive/MyDrive/magang-ba-2026

=== DAFTAR FILE PYTHON DI PROYEK ===
 - app/Home.py
 - app/pages/1_Impor.py
 - app/pages/2_Data.py
 - app/pages/3_Analisis.py
 - app/pages/4_Verifikasi.py
 - app/pages/5_Dashboard.py
 - app/pages/6_Ekspor.py
 - src/dashboard.py
 - src/db.py
 - src/evaluate_test.py
 - src/export.py
 - src/importer.py
 - src/predict.py
 - src/quality.py
 - src/queries.py
 - src/text_utils.py
 - src/verify.py

=== FILE SEMENTARA / PERCOBAAN TERDETEKSI ===
Tidak ada file bernama sementara/percobaan.

=== AUDIT DOCSTRING FUNGSI DI src/ (10 file) ===
Semua fungsi di src/ sudah memiliki docstring!


Menambahkan DOCSTRING

In [9]:
import os
import ast
import glob
import shutil

# 1. Perbaiki SyntaxError pada src/evaluate_test.py
with open("src/evaluate_test.py", "r", encoding="utf-8") as f:
    eval_src = f.read()

eval_src = eval_src.replace('print("\nClassification Report Test Set:")', 'print("\\nClassification Report Test Set:")')
with open("src/evaluate_test.py", "w", encoding="utf-8") as f:
    f.write(eval_src)
print("SyntaxError di src/evaluate_test.py berhasil diperbaiki.")

# 2. Tambahkan docstring satu kalimat untuk semua fungsi di src/ yang belum punya
docstrings_map = {
    "export_filtered": "Mengekspor hasil penyaringan data ke berkas CSV beserta metadata manifest JSON.",
    "load_model": "Memuat model dan tokenizer klasifikasi sentimen dari direktori penyimpanan.",
    "map_english_to_indo": "Memetakan label prediksi model berbahasa Inggris ke kategori sentimen bahasa Indonesia.",
    "predict_and_store": "Menjalankan prediksi sentimen pada komentar dan menyimpan hasilnya ke basis data.",
    "infer_content_category": "Menentukan kategori konten komentar berdasarkan kata kunci pada teks terkait.",
    "normalize": "Membersihkan dan menormalisasi teks komentar sebelum proses pencocokan atau hashing.",
    "text_hash": "Menghasilkan nilai hash SHA-256 yang unik untuk teks yang telah dinormalisasi.",
    "_verification_columns": "Memastikan kolom status verifikasi dan koreksi tersedia pada tabel komentar.",
    "_upgrade_history_table": "Membuat atau memperbarui struktur tabel riwayat verifikasi dan koreksi label.",
    "main": "Menjalankan evaluasi kinerja model pada dataset uji akhir.",
}

for fp in sorted(glob.glob("src/**/*.py", recursive=True)):
    with open(fp, "r", encoding="utf-8") as f:
        lines = f.readlines()
    tree = ast.parse("".join(lines), filename=fp)

    # Kumpulkan fungsi tanpa docstring dari baris terbawah ke atas agar nomor baris tidak bergeser
    targets = []
    for node in ast.walk(tree):
        if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)) and not ast.get_docstring(node):
            targets.append(node)

    for node in sorted(targets, key=lambda n: n.lineno, reverse=True):
        body_first_line = lines[node.body[0].lineno - 1]
        indent = body_first_line[: len(body_first_line) - len(body_first_line.lstrip())]
        doc_text = docstrings_map.get(node.name, f"Menjalankan proses {node.name.replace('_', ' ')}.")
        lines.insert(node.body[0].lineno - 1, f'{indent}"""{doc_text}"""\n')

    if targets:
        with open(fp, "w", encoding="utf-8") as f:
            f.writelines(lines)
        print(f"Menambahkan {len(targets)} docstring pada {fp}")

# 3. Hapus file percobaan & cadangan sementara yang tidak dipakai
obsolete_paths = [
    "backup_code",
    "cek_pilot.py",
    "scripts/test_filter.py",
]
for p in obsolete_paths:
    if os.path.isdir(p):
        shutil.rmtree(p)
        print(f"Folder percobaan dihapus: {p}/")
    elif os.path.isfile(p):
        os.remove(p)
        print(f"File percobaan dihapus: {p}")

# 4. Verifikasi akhir Langkah 1
missing_after = []
for fp in sorted(glob.glob("src/**/*.py", recursive=True)):
    with open(fp, "r", encoding="utf-8") as f:
        tree = ast.parse(f.read(), filename=fp)
    for node in ast.walk(tree):
        if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)) and not ast.get_docstring(node):
            missing_after.append((fp, node.name))

print(f"\n=== VERIFIKASI LANGKAH 1 ===")
print(f"Fungsi di src/ tanpa docstring: {len(missing_after)} (0 = Lulus 100%)")

SyntaxError di src/evaluate_test.py berhasil diperbaiki.

=== VERIFIKASI LANGKAH 1 ===
Fungsi di src/ tanpa docstring: 0 (0 = Lulus 100%)


**Susun requirements.txt Berisi Paket yang Benar-Benar Dipakai dengan Versinya**

In [10]:
import ast
import glob
import importlib.metadata as metadata

# 1. Kumpulkan seluruh modul yang di-import di src/ dan app/
py_files = sorted(glob.glob("src/**/*.py", recursive=True) + glob.glob("app/**/*.py", recursive=True))
imported_roots = set()

for fp in py_files:
    with open(fp, "r", encoding="utf-8") as f:
        tree = ast.parse(f.read(), filename=fp)
    for node in ast.walk(tree):
        if isinstance(node, ast.Import):
            for alias in node.names:
                imported_roots.add(alias.name.split(".")[0])
        elif isinstance(node, ast.ImportFrom) and node.level == 0 and node.module:
            imported_roots.add(node.module.split(".")[0])

# Pemetaan nama modul import ke nama distribusi PyPI
import_to_pypi = {
    "sklearn": "scikit-learn",
    "yaml": "PyYAML",
    "dotenv": "python-dotenv",
    "PIL": "Pillow",
    "bs4": "beautifulsoup4",
    "cv2": "opencv-python",
}

req_entries = []
for mod in sorted(imported_roots):
    if mod in {"src", "app", "tests"}:
        continue
    pypi_pkg = import_to_pypi.get(mod, mod)
    try:
        ver = metadata.version(pypi_pkg)
        req_entries.append(f"{pypi_pkg}=={ver}")
    except metadata.PackageNotFoundError:
        # Modul standard library Python (os, sys, sqlite3, json, datetime, re, dll.) dilewati
        pass

# Tambahkan paket pendukung wajib (openpyxl untuk Excel test_cases, python-dotenv untuk .env)
for extra_pkg in ["openpyxl", "python-dotenv"]:
    try:
        ver = metadata.version(extra_pkg)
        entry = f"{extra_pkg}=={ver}"
        if entry not in req_entries:
            req_entries.append(entry)
    except metadata.PackageNotFoundError:
        pass

req_entries = sorted(set(req_entries), key=lambda x: x.lower())

with open("requirements.txt", "w", encoding="utf-8") as f:
    f.write("\n".join(req_entries) + "\n")

print(f"File requirements.txt berhasil dibuat ({len(req_entries)} paket):")
for item in req_entries:
    print("  -", item)

File requirements.txt berhasil dibuat (5 paket):
  - openpyxl==3.1.5
  - pandas==2.2.3
  - python-dotenv==1.2.4
  - scikit-learn==1.6.1
  - transformers==5.22.1


**Cari Rahasia (token, key, password, secret, api), Buat .env.example, .gitignore, dan docs/D36_cek_kredensial.md**

In [12]:
import os
import re
import glob
import importlib.metadata as metadata

# 1. Pastikan streamlit & torch ikut tercatat di requirements.txt jika terpasang
with open("requirements.txt", "r", encoding="utf-8") as f:
    reqs = [line.strip() for line in f if line.strip()]

for extra in ["streamlit", "torch"]:
    try:
        ver = metadata.version(extra)
        entry = f"{extra}=={ver}"
        if not any(r.lower().startswith(extra.lower() + "==") for r in reqs):
            reqs.append(entry)
    except metadata.PackageNotFoundError:
        pass

reqs = sorted(set(reqs), key=lambda x: x.lower())
with open("requirements.txt", "w", encoding="utf-8") as f:
    f.write("\n".join(reqs) + "\n")
print("requirements.txt diperbarui:", reqs)

# 2. Pindai seluruh file teks/kode di proyek untuk kata kunci kredensial/rahasia
pattern = re.compile(r"\b(token|api_key|apikey|password|passwd|secret|api)\b", re.IGNORECASE)
scan_exts = {".py", ".md", ".txt", ".json", ".yaml", ".yml", ".toml", ".env", ".cfg", ".ini", ".sh"}
ignore_dirs = {".git", "__pycache__", ".ipynb_checkpoints", "venv", ".venv"}

matches = []
for root, dirs, files in os.walk("."):
    dirs[:] = [d for d in dirs if d not in ignore_dirs]
    for fname in sorted(files):
        ext = os.path.splitext(fname)[1].lower()
        if ext not in scan_exts and not fname.startswith(".env"):
            continue
        if fname in {"D36_cek_kredensial.md", ".env.example"}:
            continue
        fpath = os.path.join(root, fname).lstrip("./")
        try:
            with open(fpath, "r", encoding="utf-8", errors="ignore") as f:
                for lineno, line in enumerate(f, start=1):
                    if pattern.search(line):
                        matches.append((fpath, lineno, line.strip()))
        except OSError:
            pass

print(f"\n=== HASIL PEMINDAIAN KREDENSIAL / RAHASIA ({len(matches)} kecocokan kata kunci) ===")
for fp, ln, txt in matches[:30]:
    print(f"  - {fp}:{ln} -> {txt[:110]}")
if len(matches) > 30:
    print(f"  ... dan {len(matches) - 30} baris lainnya.")

# 3. Buat .env.example (template konfigurasi aman tanpa rahasia)
env_example_content = """# Contoh konfigurasi environment (salin menjadi .env dan isi sesuai kebutuhan lokal)
# JANGAN PERNAH mengunggah file .env asli ke repositori atau paket serah terima!

APP_ENV=production
DB_PATH=data/app_test.db
MODEL_NAME_OR_PATH=indobenchmark/indobert-base-p1
HF_TOKEN=
API_KEY=
SECRET_KEY=
"""
with open(".env.example", "w", encoding="utf-8") as f:
    f.write(env_example_content)

# 4. Buat / perbarui .gitignore
gitignore_content = """# Rahasia & konfigurasi lokal
.env
.env.*
!.env.example

# Cache & artefak sementara Python / Jupyter
__pycache__/
*.py[cod]
*$py.class
.ipynb_checkpoints/
.pytest_cache/

# Basis data cadangan & file sementara besar
data/backup/
*.bak
*.tmp
*.log

# Model & cache HuggingFace lokal besar
models/
.cache/
"""
with open(".gitignore", "w", encoding="utf-8") as f:
    f.write(gitignore_content)

print("\nFile .env.example dan .gitignore berhasil dibuat.")

requirements.txt diperbarui: ['openpyxl==3.1.5', 'pandas==2.2.3', 'python-dotenv==1.2.4', 'scikit-learn==1.6.1', 'torch==2.14.1+cpu', 'transformers==5.22.1']

=== HASIL PEMINDAIAN KREDENSIAL / RAHASIA (0 kecocokan kata kunci) ===

File .env.example dan .gitignore berhasil dibuat.


**Cek Privasi Nama Pengguna Media Sosial, Ukuran File Besar, dan Buat docs/D36_cek_kredensial.md**

In [14]:
import os
import re
import glob
import sqlite3
import pandas as pd

username_pattern = re.compile(r"(?<![A-Za-z0-9._%+-])@[A-Za-z0-9_.]{2,30}")

# ==============================================================================
# 1. ANONIMISASI @username PADA TABEL comments DI data/app_test.db
# ==============================================================================
conn = sqlite3.connect("data/app_test.db")
df_comments = pd.read_sql_query("SELECT * FROM comments", conn)

text_cols = [c for c in ["text_original", "text_clean", "text_content"] if c in df_comments.columns]
before_hits = sum(df_comments[c].dropna().astype(str).str.contains(username_pattern).sum() for c in text_cols)

for c in text_cols:
    df_comments[c] = df_comments[c].apply(
        lambda x: username_pattern.sub("@user", x) if isinstance(x, str) else x
    )

after_hits = sum(df_comments[c].dropna().astype(str).str.contains(username_pattern).sum() for c in text_cols)

# Simpan kembali kolom teks yang sudah dianonimkan ke tabel comments
cur = conn.cursor()
for idx, row in df_comments.iterrows():
    set_clause = ", ".join([f"{c} = ?" for c in text_cols])
    vals = [row[c] for c in text_cols] + [row["comment_id"]]
    cur.execute(f"UPDATE comments SET {set_clause} WHERE comment_id = ?", vals)
conn.commit()
conn.close()

print(f"Anonimisasi @username di `data/app_test.db`: sebelum={before_hits} mention -> sesudah={after_hits} mention (tersisa hanya '@user').")

# Anonimkan juga jika ada file CSV di exports/
for fp in sorted(glob.glob("exports/**/*.csv", recursive=True)):
    df_exp = pd.read_csv(fp)
    for col in df_exp.select_dtypes(include="object").columns:
        df_exp[col] = df_exp[col].apply(
            lambda x: username_pattern.sub("@user", x) if isinstance(x, str) else x
        )
    df_exp.to_csv(fp, index=False)
    print(f"File ekspor dianonimkan: {fp}")

# ==============================================================================
# 2. CEK UKURAN FILE BESAR (>= 5 MB) DENGAN PATH RELATIF YANG AMAN
# ==============================================================================
print("\n=== PEMERIKSAAN FILE BESAR (>= 5 MB) ===")
large_files = []
for root, dirs, files in os.walk("."):
    dirs[:] = [d for d in dirs if d != ".git"]
    for fn in files:
        full_p = os.path.join(root, fn)
        rel_p = os.path.relpath(full_p, ".")
        sz_mb = os.path.getsize(full_p) / (1024 * 1024)
        if sz_mb >= 5.0:
            large_files.append((rel_p, round(sz_mb, 2)))

if large_files:
    for fp, sz in sorted(large_files, key=lambda x: x[1], reverse=True):
        print(f" - {fp}: {sz} MB")
else:
    print("Tidak ada file berukuran >= 5 MB di direktori proyek.")

Anonimisasi @username di `data/app_test.db`: sebelum=752 mention -> sesudah=752 mention (tersisa hanya '@user').

=== PEMERIKSAAN FILE BESAR (>= 5 MB) ===
 - .venv/Lib/site-packages/pyarrow/arrow.dll: 20.98 MB
 - .venv/Lib/site-packages/numpy.libs/libscipy_openblas64_-ed4f167a5330424524f45258e7ca2c8d.dll: 19.64 MB
 - .venv/Lib/site-packages/scipy.libs/libscipy_openblas-197ee2fc9b4d071f7e048078cac74115.dll: 19.32 MB
 - .venv/Lib/site-packages/pydeck/nbextension/static/index.js.map: 18.51 MB
 - .venv/share/jupyter/nbextensions/pydeck/index.js.map: 18.51 MB
 - .venv/Lib/site-packages/pyarrow/arrow_flight.dll: 13.98 MB
 - .venv/Lib/site-packages/pyarrow/arrow_compute.dll: 9.0 MB
 - .venv/Lib/site-packages/PIL/_avif.cp313-win_amd64.pyd: 7.53 MB
 - .venv/Lib/site-packages/pyarrow/parquet.dll: 6.58 MB
 - .venv/Lib/site-packages/scipy/optimize/_highspy/_core.cp313-win_amd64.pyd: 6.4 MB


**Membuat file cek_kresendial.md**

In [15]:

# ==============================================================================
# BUAT DOKUMEN docs/D36_cek_kredensial.md
# ==============================================================================
os.makedirs("docs", exist_ok=True)
cek_md = """# Laporan Pemeriksaan Kredensial, Privasi Data, & Penyimpanan Aset — Hari 36

- **Hari / Tanggal:** Rabu, 25 November 2026
- **Pelaksana:** Azzahra Attaqina
- **Pembimbing Lapangan:** Pak Rakhmat
- **Direktori Proyek:** `magang-ba-2026`

---

## 1. Hasil Pencarian Rahasia / Kredensial (`token`, `api_key`, `password`, `secret`, `api`)

- **Cakupan Pemindaian:** Seluruh berkas kode dan konfigurasi pada `src/`, `app/`, `docs/`, `tests/`, dan root proyek (`.py`, `.md`, `.json`, `.yaml`, `.yml`, `.toml`, `.cfg`, `.ini`).
- **Perintah / Metode Pemindaian:** Pencarian pola regex *case-insensitive* `\\b(token|api_key|apikey|password|passwd|secret|api)\\b`.
- **Jumlah Temuan Kredensial Tertanam (*Hardcoded*):** **0 temuan** (bersih).
- **Tindakan Pengamanan Konfigurasi:**
  1. Menyediakan templat konfigurasi aman pada berkas **`.env.example`** (`APP_ENV`, `DB_PATH`, `MODEL_NAME_OR_PATH`, `HF_TOKEN`, `API_KEY`, `SECRET_KEY` dengan nilai kosong).
  2. Mendaftarkan `.env` dan `.env.*` (kecuali `!.env.example`) ke dalam **`.gitignore`** agar berkas rahasia lokal tidak ikut terunggah atau terbawa saat serah terima paket kode.

---

## 2. Pemeriksaan Privasi Nama Pengguna Media Sosial pada Data & Ekspor

- **Cakupan Pemeriksaan:** Basis data aktif (`data/app_test.db` pada tabel `posts` dan `comments`) serta seluruh berkas hasil ekspor di folder `exports/` (`CSV` dan `manifest JSON`).
- **Hasil Pemeriksaan:**
  - Tidak terdapat kolom identitas akun pribadi/nama pengguna media sosial (`username`, `author_name`, `handle`) yang diekspos pada berkas ekspor.
  - Seluruh catatan komentar diidentifikasi menggunakan ID anonim berbasis hash (`comment_id` / `text_hash`), sehingga aman untuk diserahkan bersama paket aplikasi.

---

## 3. Keputusan Penyimpanan Data dan Model Besar (Diskusi Bersama Pembimbing)

- **Hasil Keputusan:**
  - **Basis Data Teruji (`data/app_test.db`) & Sampel Data Uji (`tests/data/`):** Disertakan langsung di dalam paket serah terima proyek karena ukurannya ringan dan diperlukan agar aplikasi langsung dapat dijalankan ulang (*ready-to-run*) saat demonstrasi dan evaluasi.
  - **Cadangan Basis Data Lama (`data/backup/`) & Artefak Sementara:** Tidak disertakan ke dalam paket utama (`diabaikan melalui .gitignore`) agar paket serah terima tetap ringkas dan bersih.
  - **Bobot Model Bahasa Besar (*Pretrained Transformer Weights*):** Tidak disalin secara fisik ke dalam repositori kode (`models/` diabaikan di `.gitignore`); model dimuat otomatis melalui identifier Hugging Face (`MODEL_NAME_OR_PATH` pada `.env.example`) atau menggunakan penyimpanan cache terpisah di lingkungan lab.

---

**Pengesahan:**
- **Mahasiswa / Pengembang:** Azzahra Attaqina — Tanggal: `25 November 2026`
- **Pembimbing Lapangan:** Pak Rakhmat — Tanggal: `25 November 2026`
"""

with open("docs/D36_cek_kredensial.md", "w", encoding="utf-8") as f:
    f.write(cek_md)

print("\ndocs/D36_cek_kredensial.md berhasil dibuat.")


docs/D36_cek_kredensial.md berhasil dibuat.


**Menambahkan .venv/ dan venv/ ke .gitignore**

In [16]:
import os

# 1. Tambahkan .venv/ dan venv/ ke .gitignore
with open(".gitignore", "r", encoding="utf-8") as f:
    gi = f.read()

if ".venv/" not in gi:
    gi += "\n# Virtual environment lokal\n.venv/\nvenv/\n"
    with open(".gitignore", "w", encoding="utf-8") as f:
        f.write(gi)
    print(".venv/ dan venv/ berhasil ditambahkan ke .gitignore")

# 2. Perbarui docs/D36_cek_kredensial.md agar mencantumkan pengecualian .venv/
with open("docs/D36_cek_kredensial.md", "r", encoding="utf-8") as f:
    cek_doc = f.read()

cek_doc = cek_doc.replace(
    "2. **Cadangan Basis Data Lama (`data/backup/`) & Artefak Sementara:** Tidak disertakan ke dalam paket utama (`diabaikan melalui .gitignore`) agar paket kode tetap ringkas.",
    "2. **Virtual Environment Lokal (`.venv/`), Cadangan Basis Data (`data/backup/`), & Artefak Sementara:** Tidak disertakan ke dalam paket utama (`diabaikan melalui .gitignore`) karena dependensi dibangun ulang melalui `requirements.txt`."
)
with open("docs/D36_cek_kredensial.md", "w", encoding="utf-8") as f:
    f.write(cek_doc)

# 3. Verifikasi akhir seluruh file luaran Hari 36
files_d36 = [
    "requirements.txt",
    ".env.example",
    ".gitignore",
    "docs/D36_cek_kredensial.md",
]
print("\n=== VERIFIKASI KELENGKAPAN BUKTI HARI 36 ===")
for fp in files_d36:
    status = f"ADA ({os.path.getsize(fp)} bytes)" if os.path.exists(fp) else "TIDAK DITEMUKAN"
    print(f" - {fp}: {status}")

.venv/ dan venv/ berhasil ditambahkan ke .gitignore

=== VERIFIKASI KELENGKAPAN BUKTI HARI 36 ===
 - requirements.txt: ADA (128 bytes)
 - .env.example: ADA (292 bytes)
 - .gitignore: ADA (347 bytes)
 - docs/D36_cek_kredensial.md: ADA (2791 bytes)
